# 02: Preprocessing Pipeline

This notebook implements the full preprocessing pipeline for the credit card default prediction task.
All transformations are fit exclusively on the training set and applied to both train and test sets to prevent data leakage.

Inputs: raw train/test splits saved from  `01_EDA.ipynb` → `data/processed/` 

Outputs: preprocessed feature matrices saved back to `data/processed/artefacts/`

Imports:

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import math
import joblib
from imblearn.under_sampling import RandomUnderSampler



## Step 1: Load Saved Splits

In [2]:
data_dir = os.path.join(os.path.dirname(os.getcwd()), "data", "processed")

X_train = pd.read_csv(os.path.join(data_dir, "X_train.csv"))
X_test  = pd.read_csv(os.path.join(data_dir, "X_test.csv"))
y_train = pd.read_csv(os.path.join(data_dir, "y_train.csv")).squeeze()
y_test  = pd.read_csv(os.path.join(data_dir, "y_test.csv")).squeeze()

print("Shapes loaded:")
print(f"  X_train : {X_train.shape}")
print(f"  X_test  : {X_test.shape}")
print(f"  y_train : {y_train.shape}")
print(f"  y_test  : {y_test.shape}")

Shapes loaded:
  X_train : (24000, 23)
  X_test  : (6000, 23)
  y_train : (24000,)
  y_test  : (6000,)


The dataset contains 24,000 training observations and 6,000 test observations across 23 features, following an 80/20 split consistent with standard practice for datasets of this size. The class distribution immediately flags a structural challenge: 77.9% of clients did not default (class 0) and 22.1% defaulted (class 1). While this imbalance is not extreme, accuracy is a deeply misleading metric for skewed datasets as a classifier that blindly predicts no default for every observation would achieve 78% accuracy while providing zero predictive value. 

This motivates the use of **ROC-AUC** and **F1 score** as the primary evaluation metrics throughout this project, as both account for class imbalance by considering the trade-off between true positive and false positive rates, and between precision and recall respectively. 

The imbalance also motivates SMOTE oversampling in Step 5. SMOTE is applied only to the training portion, the validation and test sets are left at the natural 22% default rate so that evaluation metrics reflect how the model would actually perform in the real world, not on artificially balanced data.

In [3]:
X_train.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6
0,160000,2,2,2,33,2,2,3,2,0,...,168541,164310,162681,163005,15000,0,0,6100,12300,6100
1,150000,2,1,2,34,1,-1,-1,-2,-2,...,0,0,0,0,53,0,0,0,0,0
2,10000,1,2,1,50,1,2,0,0,0,...,8446,8067,8227,8400,2,1281,1134,294,305,1000
3,220000,2,1,2,29,0,0,0,0,0,...,215139,218513,131660,134346,9100,9000,7887,4800,4900,6000
4,310000,2,1,2,32,1,-2,-1,0,0,...,326,326,-235,-235,0,326,0,0,0,1200


In [4]:
# Verifying column names match between train and test sets
assert list(X_train.columns) == list(X_test.columns), \
    f"Column mismatch: {set(X_train.columns) ^ set(X_test.columns)}"
print(" Column names match between X_train and X_test")
print(f"   Columns ({len(X_train.columns)}): {list(X_train.columns)}")

 Column names match between X_train and X_test
   Columns (23): ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']


In [5]:
print("y_train class distribution:")
print(y_train.value_counts())
print()
print("y_train class proportions:")
print(y_train.value_counts(normalize=True).round(4))

y_train class distribution:
default payment next month
0    18691
1     5309
Name: count, dtype: int64

y_train class proportions:
default payment next month
0    0.7788
1    0.2212
Name: proportion, dtype: float64


## Step 2: Recode Undocumented Category Values

EDA identified category codes not described in the original dataset documentation (Yeh & Lien, 2009):
- **EDUCATION**: codes 0, 5, 6 are undocumented (n=12, 235, 42 in training set). Remapped to 4 ("others").
- **MARRIAGE**: code 0 is undocumented (n=47 in training set). Remapped to 3 ("others").

The same mapping is applied to both train and test sets using rules defined from training set findings only.

In [6]:
# Per Yeh & Lien (2009), valid EDUCATION codes are 1=graduate school,
# 2=university, 3=high school, 4=others. Codes 0 (n=12), 5 (n=235), 
# 6 (n=42) have no documented meaning and are consolidated into 4 ("others").
edu_remap = {0: 4, 5: 4, 6: 4}
X_train["EDUCATION"] = X_train["EDUCATION"].replace(edu_remap)
X_test["EDUCATION"]  = X_test["EDUCATION"].replace(edu_remap)

# Per Yeh & Lien (2009), valid MARRIAGE codes are 1=married, 2=single,
# 3=others. Code 0 (n=47) has no documented meaning and is remapped to 3.
mar_remap = {0: 3}
X_train["MARRIAGE"] = X_train["MARRIAGE"].replace(mar_remap)
X_test["MARRIAGE"]  = X_test["MARRIAGE"].replace(mar_remap)

print("EDUCATION value_counts after remapping:")
print("  train:")
print(X_train["EDUCATION"].value_counts().sort_index().to_string())
print("  test:")
print(X_test["EDUCATION"].value_counts().sort_index().to_string())
print()
print("MARRIAGE value_counts after remapping:")
print("  train:")
print(X_train["MARRIAGE"].value_counts().sort_index().to_string())
print("  test:")
print(X_test["MARRIAGE"].value_counts().sort_index().to_string())
print()
# Assertions: confirm no undocumented codes remain
assert set(X_train["EDUCATION"].unique()).issubset({1, 2, 3, 4}), "Unexpected EDUCATION codes in train"
assert set(X_test["EDUCATION"].unique()).issubset({1, 2, 3, 4}),  "Unexpected EDUCATION codes in test"
assert set(X_train["MARRIAGE"].unique()).issubset({1, 2, 3}),     "Unexpected MARRIAGE codes in train"
assert set(X_test["MARRIAGE"].unique()).issubset({1, 2, 3}),      "Unexpected MARRIAGE codes in test"
print("All assertions passed, no undocumented codes remain.")

EDUCATION value_counts after remapping:
  train:
EDUCATION
1     8455
2    11256
3     3903
4      386
  test:
EDUCATION
1    2130
2    2774
3    1014
4      82

MARRIAGE value_counts after remapping:
  train:
MARRIAGE
1    10892
2    12806
3      302
  test:
MARRIAGE
1    2767
2    3158
3      75

All assertions passed, no undocumented codes remain.


The remapping consolidates all observations into the documented category space defined by Yeh & Lien (2009). The combined undocumented Education codes (0, 5, 6) represent 289 training observations, approximately 1.2% of the dataset which are too few to treat as a meaningful separate category and too ambiguous to interpret without documentation. Consolidating them into category 4 ("others") is the most defensible approach as it avoids introducing spurious information while retaining all observations in the dataset.

 The same logic applies to MARRIAGE code 0 (n=47, 0.2%). Data cleaning is a prerequisite to reliable modelling, noting that poor-quality inputs, including inconsistently coded categories constitute a form of noise that degrades model performance in ways that are difficult to diagnose downstream.
 
 Crucially, the remapping rules are defined solely from training set findings and applied identically to the test set without consulting test-set distributions, preserving the integrity of final evaluation.

## Step 3:  Feature Engineering

Seven derived features are constructed from the raw variables:

| Feature | Definition | Rationale | Design Choice |
|---|---|---|---|
| `UTIL_RATE` | `BILL_AMT1 / LIMIT_BAL`, clipped [0, 1] | Most-recent credit utilisation; negatives (overpayments) treated as 0 ( if you've overpaid your bill, you're not under any financial pressure at all, so your utilisation is zero.)| BILL_AMT1 used (not an average) because we want the client's current position at prediction time. Negatives clipped to 0, overpayments mean zero utilisation, not negative. LIMIT_BAL=0 set to 0 to avoid division by zero. |
| `AVG_PAY_STATUS` | Mean of `PAY_0–PAY_6` | Average delinquency level over 6 months | Mean across all 6 months to smooth out one-off anomalies and capture general repayment behaviour over time. |
| `MAX_DLQ` | Max of `PAY_0–PAY_6` | Worst single delinquency episode | Max rather than mean because a single severe delinquency is itself a risk signal that averaging would conceal. Retained alongside AVG_PAY_STATUS because the two capture different things. |
| `TOTAL_BILL` | Sum of `BILL_AMT1–6` | Total outstanding balance over 6 months | Sum rather than mean  total exposure is more meaningful than an average monthly figure for assessing overall debt burden. |
| `TOTAL_PAY` | Sum of `PAY_AMT1–6` | Total amount paid over 6 months | Sum to match TOTAL_BILL both need to be on the same scale for PAY_RATIO to be interpretable. |
| `PAY_RATIO` | `TOTAL_PAY / (TOTAL_BILL + 1)`, clipped [0, 5] | Repayment coverage ratio, how much of total debt was paid back | +1 added to denominator to avoid division by zero when TOTAL_BILL=0. Clipped at 5 to prevent extreme outliers from clients who massively overpaid. |
| `BILL_TREND` | `BILL_AMT1 − BILL_AMT6` | Debt trajectory: positive = growing, negative = shrinking | Most recent minus oldest to get a direction of travel. Positive = debt growing over 6 months, negative = debt shrinking. Simple difference chosen over slope for interpretability. |

`engineer_features()` is applied to both X_train and X_test so that both sets have the same 30 columns before entering the pipeline. The model is trained on these features and therefore expects them to exist at evaluation time too. This is safe to do simultaneously on both sets because all 7 features are pure row-level arithmetic (divisions, sums, differences) requiring no statistics learned from the training data, therefore there is no leakage risk.

In [7]:
pay_cols     = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
bill_cols    = ["BILL_AMT1", "BILL_AMT2", "BILL_AMT3", "BILL_AMT4", "BILL_AMT5", "BILL_AMT6"]
pay_amt_cols = ["PAY_AMT1", "PAY_AMT2", "PAY_AMT3", "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"]

def engineer_features(df):
    df = df.copy()

    # UTIL_RATE: clip negative BILL_AMT1 to 0, cap at 1, LIMIT_BAL==0 -> 0
    bill1_clipped = df["BILL_AMT1"].clip(lower=0)
    limit_safe = df["LIMIT_BAL"].replace(0, np.nan)
    df["UTIL_RATE"] = (bill1_clipped / limit_safe).clip(upper=1).fillna(0)

    # AVG_PAY_STATUS: mean delinquency over 6 months
    df["AVG_PAY_STATUS"] = df[pay_cols].mean(axis=1)

    # MAX_DLQ: worst single delinquency month
    df["MAX_DLQ"] = df[pay_cols].max(axis=1)

    # TOTAL_BILL / TOTAL_PAY: 6-month sums
    df["TOTAL_BILL"] = df[bill_cols].sum(axis=1)
    df["TOTAL_PAY"]  = df[pay_amt_cols].sum(axis=1)

    # PAY_RATIO: repayment coverage, clipped [0, 5]
    df["PAY_RATIO"] = (df["TOTAL_PAY"] / (df["TOTAL_BILL"] + 1)).clip(lower=0, upper=5)

    # BILL_TREND: positive = debt growing, negative = debt shrinking
    df["BILL_TREND"] = df["BILL_AMT1"] - df["BILL_AMT6"]

    return df

X_train = engineer_features(X_train)
X_test  = engineer_features(X_test)

# Verification
new_feats = ["UTIL_RATE", "AVG_PAY_STATUS", "MAX_DLQ",
             "TOTAL_BILL", "TOTAL_PAY", "PAY_RATIO", "BILL_TREND"]

print(f"X_train shape after feature engineering: {X_train.shape}")
print(f"X_test  shape after feature engineering: {X_test.shape}")

for feat in new_feats:
    nan_train = X_train[feat].isna().sum()
    nan_test  = X_test[feat].isna().sum()
    print(f"── {feat}  (NaNs — train: {nan_train}, test: {nan_test})")
    print(X_train[feat].describe().round(4).to_string())
    print()


X_train shape after feature engineering: (24000, 30)
X_test  shape after feature engineering: (6000, 30)
── UTIL_RATE  (NaNs — train: 0, test: 0)
count    24000.0000
mean         0.4145
std          0.3868
min          0.0000
25%          0.0218
50%          0.3149
75%          0.8288
max          1.0000

── AVG_PAY_STATUS  (NaNs — train: 0, test: 0)
count    24000.0000
mean        -0.1834
std          0.9780
min         -2.0000
25%         -0.8333
50%          0.0000
75%          0.0000
max          6.0000

── MAX_DLQ  (NaNs — train: 0, test: 0)
count    24000.0000
mean         0.4407
std          1.3392
min         -2.0000
25%          0.0000
50%          0.0000
75%          2.0000
max          8.0000

── TOTAL_BILL  (NaNs — train: 0, test: 0)
count    2.400000e+04
mean     2.689717e+05
std      3.781240e+05
min     -3.362590e+05
25%      2.881525e+04
50%      1.262690e+05
75%      3.413595e+05
max      5.263883e+06

── TOTAL_PAY  (NaNs — train: 0, test: 0)
count    2.400000e+04
mean

- Three column groups are defined first, the 6 PAY columns, 6 BILL_AMT columns, and 6 PAY_AMT columns. So they can be referenced by name rather than repeated throughout the function.

- engineer_features() then creates the 7 new columns on whatever DataFrame is passed in, using df.copy() so the original is never modified. The function is then applied to both X_train and X_test.

- The verification loop at the bottom prints the shape of each set after engineering to confirm 7 new columns were added, then prints NaN counts and .describe() for each new feature to confirm no missing values were introduced and the distributions look sensible.

**Feature Engineering Explanations**:

The seven engineered features compress 18 raw columns into behavioural summaries that are more directly interpretable and potentially more predictive than the individual monthly figures alone. Géron (2019, Ch. 2) explicitly recommends experimenting with attribute combinations as part of data preparation, noting that derived features often carry stronger predictive signal than the raw inputs from which they are computed. Each feature here encodes a distinct behavioural dimension of credit risk.

- UTIL_RATE captures how close a client is to their credit limit at the most recent month, high utilisation is a widely recognised indicator of financial stress.

- AVG_PAY_STATUS and MAX_DLQ together characterise repayment behaviour from two complementary angles: the average smooths month-to-month noise to reflect general behaviour, while the maximum captures the single worst episode that a smooth average could conceal. A client with mostly on-time payments but one severe delinquency would show a low AVG_PAY_STATUS but a high MAX_DLQ. Therefore retaining both features allows the model to distinguish this profile from a consistently delinquent one. 

- PAY_RATIO expresses whether the client is keeping up with their outstanding balance in aggregate, values near zero indicate a client paying very little relative to what they owe. 

- BILL_TREND provides a trajectory signal absent from any individual month's balance, distinguishing a client with stable debt from one whose balance is actively growing.



In [8]:
# Check multicollinearity among engineered features
new_feats = ["UTIL_RATE", "AVG_PAY_STATUS", "MAX_DLQ", 
             "TOTAL_BILL", "TOTAL_PAY", "PAY_RATIO", "BILL_TREND"]

corr_matrix = X_train[new_feats].corr().round(3)
print("Correlation matrix of engineered features:")
print(corr_matrix.to_string())
print()

# Flag any highly correlated pairs (|r| > 0.8)
print("Highly correlated pairs (|r| > 0.8):")
found = False
for i in range(len(new_feats)):
    for j in range(i+1, len(new_feats)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.8:
            print(f"  {new_feats[i]} vs {new_feats[j]}: r = {r}")
            found = True
if not found:
    print("  None found no multicollinearity concern")

Correlation matrix of engineered features:
                UTIL_RATE  AVG_PAY_STATUS  MAX_DLQ  TOTAL_BILL  TOTAL_PAY  PAY_RATIO  BILL_TREND
UTIL_RATE           1.000           0.495    0.278       0.517     -0.024     -0.486       0.389
AVG_PAY_STATUS      0.495           1.000    0.806       0.282     -0.078     -0.438       0.037
MAX_DLQ             0.278           0.806    1.000       0.093     -0.116     -0.274      -0.031
TOTAL_BILL          0.517           0.282    0.093       1.000      0.340     -0.300       0.322
TOTAL_PAY          -0.024          -0.078   -0.116       0.340      1.000      0.141      -0.049
PAY_RATIO          -0.486          -0.438   -0.274      -0.300      0.141      1.000      -0.170
BILL_TREND          0.389           0.037   -0.031       0.322     -0.049     -0.170       1.000

Highly correlated pairs (|r| > 0.8):
  AVG_PAY_STATUS vs MAX_DLQ: r = 0.806


The multicollinearity check flagged one high correlation: AVG_PAY_STATUS vs MAX_DLQ (r = 0.806). This is expected given both are derived from the same six PAY columns. 
Both features are retained because they capture distinct signals, AVG_PAY_STATUS reflects general repayment behaviour over time while MAX_DLQ captures the single worst delinquency episode, which averaging would conceal. However, for linear models such as logistic regression, this correlation can destabilise coefficient estimates and make individual feature contributions difficult to interpret reliably. This is 
noted as a limitation to monitor during modelling if logistic regression is used, dropping one of the two features or applying regularisation should be considered.

In [9]:

scale_cols = ["UTIL_RATE", "TOTAL_BILL"]

scaler_util_bill = StandardScaler()
scaler_util_bill.fit(X_train[scale_cols])  # fit on train only

X_train[scale_cols] = scaler_util_bill.transform(X_train[scale_cols])
X_test[scale_cols]  = scaler_util_bill.transform(X_test[scale_cols])

print("Scaler means  (from X_train):", dict(zip(scale_cols, scaler_util_bill.mean_.round(4))))
print("Scaler stdevs (from X_train):", dict(zip(scale_cols, scaler_util_bill.scale_.round(4))))
print()
print("Post-scaling X_train stats:")
print(X_train[scale_cols].describe().round(4).to_string())
print()
print("Post-scaling X_test stats:")
print(X_test[scale_cols].describe().round(4).to_string())


Scaler means  (from X_train): {'UTIL_RATE': np.float64(0.4145), 'TOTAL_BILL': np.float64(268971.7039)}
Scaler stdevs (from X_train): {'UTIL_RATE': np.float64(0.3868), 'TOTAL_BILL': np.float64(378116.1012)}

Post-scaling X_train stats:
        UTIL_RATE  TOTAL_BILL
count  24000.0000  24000.0000
mean      -0.0000     -0.0000
std        1.0000      1.0000
min       -1.0715     -1.6006
25%       -1.0152     -0.6351
50%       -0.2573     -0.3774
75%        1.0713      0.1914
max        1.5139     13.2100

Post-scaling X_test stats:
       UTIL_RATE  TOTAL_BILL
count  6000.0000   6000.0000
mean      0.0027      0.0118
std       1.0027      1.0190
min      -1.0715     -1.3977
25%      -1.0128     -0.6365
50%      -0.2687     -0.3759
75%       1.0829      0.2097
max       1.5139      8.2777


The scaler must be fit on the training data only and then applied to both sets. Fitting on the test set or on the combined train and test would allow test distribution statistics to influence the scaling parameters, constituting data leakage. 

The post-scaling output confirms this was done correctly: X_train shows mean ≈ 0 and std ≈ 1 exactly, while X_test shows mean ≈ 0.003 and std ≈ 1.003. These are small but non zero deviations that are expected and desirable, confirming the test set was transformed using training parameters only and was never used in fitting.

## Step 4: ColumnTransformer Preprocessing Pipeline

A single `ColumnTransformer` encodes all transformations in one reproducible object.
It is **fit on `X_train` only** and applied identically to both sets.

| Group | Columns | Transformer |
|---|---|---|
| `num_cols` | `LIMIT_BAL`, `AGE`, `BILL_AMT1–6`, `PAY_AMT1–6`, `AVG_PAY_STATUS`, `MAX_DLQ`, `TOTAL_PAY`, `PAY_RATIO`, `BILL_TREND` | `StandardScaler` |
| `ord_cols` | `PAY_0`, `PAY_2–PAY_6` | `StandardScaler` |
| `cat_cols` | `SEX`, `EDUCATION`, `MARRIAGE` | `OneHotEncoder(drop='first')` |
| `pass_cols` | `UTIL_RATE`, `TOTAL_BILL` | passthrough (already scaled) |

`verbose_feature_names_out=False` keeps clean column names (no transformer prefix as ColumnTransformer adds a prefix based on which transformer processed it).

In [10]:

num_cols = [
    "LIMIT_BAL", "AGE",
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3", "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
    "PAY_AMT1",  "PAY_AMT2",  "PAY_AMT3",  "PAY_AMT4",  "PAY_AMT5",  "PAY_AMT6",
    "AVG_PAY_STATUS", "MAX_DLQ", "TOTAL_PAY", "PAY_RATIO", "BILL_TREND",
]
ord_cols  = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
cat_cols  = ["SEX", "EDUCATION", "MARRIAGE"]
pass_cols = ["UTIL_RATE", "TOTAL_BILL"]  # already scaled above

ct = ColumnTransformer(
    transformers=[
        ("num",  StandardScaler(),                               num_cols),
        ("ord",  StandardScaler(),                               ord_cols),
        ("cat",  OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"), cat_cols),
        ("pass", "passthrough",                                  pass_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)


ct.fit(X_train)

X_train_proc = pd.DataFrame(
    ct.transform(X_train),
    columns=ct.get_feature_names_out(),
    index=X_train.index,
)
X_test_proc = pd.DataFrame(
    ct.transform(X_test),
    columns=ct.get_feature_names_out(),
    index=X_test.index,
)

print("Output columns:")
print(list(X_train_proc.columns))
print()
print(f"X_train_proc shape : {X_train_proc.shape}")
print(f"X_test_proc  shape : {X_test_proc.shape}")
print()
print(f"NaNs in X_train_proc : {X_train_proc.isna().sum().sum()}")
print(f"NaNs in X_test_proc  : {X_test_proc.isna().sum().sum()}")
print()
print("X_train_proc sample stats (first 5 cols):")
print(X_train_proc.iloc[:, :5].describe().round(4).to_string())


Output columns:
['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'AVG_PAY_STATUS', 'MAX_DLQ', 'TOTAL_PAY', 'PAY_RATIO', 'BILL_TREND', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'SEX_2', 'EDUCATION_2', 'EDUCATION_3', 'EDUCATION_4', 'MARRIAGE_2', 'MARRIAGE_3', 'UTIL_RATE', 'TOTAL_BILL']

X_train_proc shape : (24000, 33)
X_test_proc  shape : (6000, 33)

NaNs in X_train_proc : 0
NaNs in X_test_proc  : 0

X_train_proc sample stats (first 5 cols):
        LIMIT_BAL         AGE   BILL_AMT1   BILL_AMT2   BILL_AMT3
count  24000.0000  24000.0000  24000.0000  24000.0000  24000.0000
mean       0.0000     -0.0000      0.0000     -0.0000      0.0000
std        1.0000      1.0000      1.0000      1.0000      1.0000
min       -1.2151     -1.5696     -2.9477     -1.6769     -1.5843
25%       -0.9062     -0.8083     -0.6474     -0.6497     -0.6463
50%       -0.2113     -0.

The ColumnTransformer applies different preprocessing steps to separate column groups while producing a single unified feature matrix. As recommended in Hands‑On Machine Learning with Scikit‑Learn, Keras, and TensorFlow, preprocessing steps are encapsulated within a pipeline and fit only on the training data. The fitted transformations are then applied unchanged to validation and test sets, ensuring consistent feature processing and preventing data leakage

The treatment of each feature group reflects deliberate choices grounded in the nature of each variable. 

The PAY columns (PAY_0 through PAY_6) are treated as ordinal rather than nominal because they encode a meaningful ordered sequence: -2 (no consumption) through to 8 (eight months delayed). StandardScaler preserves this ordering while standardising the scale, which is appropriate since the columns are already numerically encoded with a clear direction. 

SEX, EDUCATION, and MARRIAGE are one-hot encoded with drop='first' as including all dummy columns from a one-hot encoding introduces perfect multicollinearity since one column is always perfectly predictable from the others, making the first-category drop essential for well-behaved linear models. 

UTIL_RATE and TOTAL_BILL pass through unchanged since they were already scaled in the previous step and including them in the scaler group would apply StandardScaler a second time, corrupting the previously learned standardisation.

The final feature matrix of 33 columns consists of 19 scaled numeric features, 6 scaled ordinal features, 6 OHE dummies (SEX_2, EDUCATION_2/3/4, MARRIAGE_2/3), and  passthrough features (UTIL_RATE and TOTAL_BILL, already scaled in Step 3b) features. Zero NaNs in both splits confirms the pipeline introduced no data quality issues during transformation.

**Note**: By default, OneHotEncoder crashes if it encounters a category at inference time that it never saw during training. handle_unknown=ignore prevents this  instead of crashing, it outputs zeros for the unseen category and the model can still produce a prediction. This dataset already contained undocumented category codes, making this a real risk. The parameter has no effect on current train/test data (confirmed by identical output after adding it)  


### **Step 5 A: Validation Split and Class Imbalance Handling**

##### Imbalance handling strategy

The training set contains ~78% non-default (class 0) and ~22% default (class 1).  
Three strategies were considered:

- **SMOTE - Selected:**  
  Generates synthetic minority samples via interpolation between neighbouring minority observations. This introduces new synthetic points rather than simple duplication, reducing overfitting risk compared with random oversampling. Applied only to the training subset after the validation split to ensure synthetic samples do not influence validation evaluation.

- **Random oversampling:**  
  Duplicates existing minority samples. Simpler but adds no new information and increases overfitting risk. Rejected in favour of SMOTE.

- **Class weights only:**  
  Passes class_weight='balanced' to classifiers without resampling. Acceptable for mild imbalance but often less effective than resampling for a 78/22 split. Rejected.



##### Validation split source

Two options were considered:

- **From X_train_proc (post-pipeline) - Selected:**  
  The ColumnTransformer was already fitted on the full X_train in Step 4, so splitting post-pipeline preserves consistency between transformed datasets.

- **From raw X_train (pre-pipeline):**  
  Technically the gold standard (fit pipeline on train-only, apply to validation separately) but impractical once the pipeline had already been fitted. This is noted as a methodological limitation.


Ideally, the validation split would be created before fitting the preprocessing pipeline so that preprocessing parameters are estimated using the training subset only. In this implementation, the ColumnTransformer was fitted on the full training data before the validation split, meaning validation observations influenced preprocessing parameters such as scaling statistics.

While labels remain unseen and the practical impact is minimal, this is noted as a methodological limitation. Importantly, the test set remains completely untouched by both preprocessing fitting and resampling procedures, ensuring that the final model evaluation is conducted on fully unseen data.

In [11]:

#  1. Stratified val split (15%) 
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_proc, y_train,
    test_size=0.15,
    stratify=y_train,
    random_state=42,
)

print("Before SMOTE:")
print(f"  X_tr  : {X_tr.shape}    X_val : {X_val.shape}")
print(f"  y_tr class counts  : {dict(y_tr.value_counts().sort_index())}")
print(f"  y_val class counts : {dict(y_val.value_counts().sort_index())}")
print(f"  y_tr  minority % : {y_tr.mean():.4f}")
print(f"  y_val minority % : {y_val.mean():.4f}")
print()

#  2. SMOTE on training portion only
smote = SMOTE(random_state=42, sampling_strategy=0.5)
X_tr_res, y_tr_res = smote.fit_resample(X_tr, y_tr)

# Restore DataFrame column names
X_tr_res = pd.DataFrame(X_tr_res, columns=X_tr.columns)
y_tr_res = pd.Series(y_tr_res, name=y_train.name)

print("After SMOTE:")
print(f"  X_tr_res : {X_tr_res.shape}")
print(f"  y_tr_res class counts : {dict(y_tr_res.value_counts().sort_index())}")
print(f"  y_tr_res minority %   : {y_tr_res.mean():.4f}")
print()
print(f"NaNs in X_tr_res : {X_tr_res.isna().sum().sum()}")
print(f"NaNs in X_val    : {X_val.isna().sum().sum()}")


Before SMOTE:
  X_tr  : (20400, 33)    X_val : (3600, 33)
  y_tr class counts  : {0: np.int64(15887), 1: np.int64(4513)}
  y_val class counts : {0: np.int64(2804), 1: np.int64(796)}
  y_tr  minority % : 0.2212
  y_val minority % : 0.2211

After SMOTE:
  X_tr_res : (23830, 33)
  y_tr_res class counts : {0: np.int64(15887), 1: np.int64(7943)}
  y_tr_res minority %   : 0.3333

NaNs in X_tr_res : 0
NaNs in X_val    : 0


The order of operations here is critical and deliberate. The validation set is carved out first from X_train_proc using a stratified split, then SMOTE is applied only to the remaining training portion (X_tr).  The test set  and by extension any held out evaluation set  must never be used in training or preprocessing decisions. If SMOTE had been applied before splitting, synthetic minority-class observations would have leaked into the validation set. Evaluation metrics would then reflect performance on artificial data points rather than real unseen observations, producing an optimistically biased and misleading estimate of generalisation performance. By keeping the validation set composed entirely of real observations at the natural 22.11% default rate, it accurately represents the deployment distribution.
Stratified sampling is used for the validation split, as  our dataset has a meaningful class imbalance and simple random sampling could produce a validation set with an unrepresentative class ratio by chance. The output confirms stratification worked correctly with:
 y_tr minority rate 0.2212, y_val minority rate 0.2211  a difference of just 0.01%.

SMOTE generates synthetic minority-class observations by interpolating between real minority-class neighbours in feature space, rather than simply duplicating existing ones. This exposes the model to a broader variety of minority-class patterns than random oversampling would, reducing overfitting risk to specific training observations. The sampling_strategy=0.5 targets a 67/33 balance, class 1 is set to half of class 0, generating ~3,430 synthetic samples on top of 4,513 real minority observations. A true 50/50 balance was deliberately avoided as it would require ~12,000 synthetic samples, meaning 75% of minority class training data would be synthetic rather than real. The true default rate of ~22% is deliberately preserved in both the validation and test sets so that evaluation reflects real-world conditions, even though training was performed on balanced data.

## Step 5b: Undersampling Baseline Dataset

A second training set is created using **random undersampling** instead of SMOTE, to allow a fair comparison of sampling strategies on the same four candidate models.

**Ratio target: 67/33 (minority : majority ≈ 1:2)** — matching the effective class ratio produced by the SMOTE-resampled dataset. This is achieved by retaining all minority-class observations and randomly downsampling the majority class to twice the minority count (sampling_strategy=0.5 in RandomUnderSampler), resulting in class proportions consistent with the SMOTE configuration ensuring fairness of comparison of sampling strategies.

Observations are removed randomly (not sequentially) to avoid introducing ordering bias
or accidentally discarding structured patterns from the data.

The source is X_tr / y_tr which is the the same pre-SMOTE split used in Step 5, so the only variable that differs between the two experiments is the sampling method itself.
Any downstream performance difference between `X_tr_res` (SMOTE) and `X_tr_us` (undersampled) is therefore attributable solely to the sampling strategy.

In [12]:


print("Before undersampling (X_tr):")
print(f"  Shape        : {X_tr.shape}")
print(f"  Class counts : {dict(y_tr.value_counts().sort_index())}")
print(f"  Minority %   : {y_tr.mean():.4f}")
print()

rus = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
X_tr_us_arr, y_tr_us_arr = rus.fit_resample(X_tr, y_tr)

X_tr_us = pd.DataFrame(X_tr_us_arr, columns=X_tr.columns)
y_tr_us = pd.Series(y_tr_us_arr, name=y_train.name)

print("After undersampling (X_tr_us):")
print(f"  Shape        : {X_tr_us.shape}")
print(f"  Class counts : {dict(y_tr_us.value_counts().sort_index())}")
print(f"  Minority %   : {y_tr_us.mean():.4f}")
print()
print(f"NaNs in X_tr_us : {X_tr_us.isna().sum().sum()}")


Before undersampling (X_tr):
  Shape        : (20400, 33)
  Class counts : {0: np.int64(15887), 1: np.int64(4513)}
  Minority %   : 0.2212

After undersampling (X_tr_us):
  Shape        : (13539, 33)
  Class counts : {0: np.int64(9026), 1: np.int64(4513)}
  Minority %   : 0.3333

NaNs in X_tr_us : 0


In [13]:
artefact_dir = os.path.join(os.path.dirname(os.getcwd()), "data", "processed", "artefacts")

X_tr_us.to_csv(os.path.join(artefact_dir, "X_tr_us.csv"), index=False)
y_tr_us.to_frame().to_csv(os.path.join(artefact_dir, "y_tr_us.csv"), index=False)

print(f"Saved to: {artefact_dir}")
print("  X_tr_us.csv")
print("  y_tr_us.csv")


Saved to: c:\Users\polym\Desktop\Term_2\Predictive\Assignment\Code\PA\data\processed\artefacts
  X_tr_us.csv
  y_tr_us.csv


## Step 6: Data Validation

Seven automated checks confirm the preprocessing pipeline produced clean, consistent outputs before any model is trained.

In [14]:


results = []

def check(label, passed):
    status = "PASS" if passed else "FAIL"
    symbol = "✅" if passed else "❌"
    results.append((label, passed))
    print(f"{symbol} {status}  |  {label}")

splits = {"X_tr_res": X_tr_res, "X_val": X_val, "X_test_proc": X_test_proc}

# 1. No NaNs in any split
nan_counts = {name: df.isna().sum().sum() for name, df in splits.items()}
check(
    f"No NaNs  (X_tr_res={nan_counts['X_tr_res']}, X_val={nan_counts['X_val']}, X_test_proc={nan_counts['X_test_proc']})",
    all(v == 0 for v in nan_counts.values()),
)

# 2. No infinite values in any split
inf_counts = {name: np.isinf(df.values).sum() for name, df in splits.items()}
check(
    f"No infinities  (X_tr_res={inf_counts['X_tr_res']}, X_val={inf_counts['X_val']}, X_test_proc={inf_counts['X_test_proc']})",
    all(v == 0 for v in inf_counts.values()),
)

# 3. Identical feature column count across all three splits
col_counts = {name: df.shape[1] for name, df in splits.items()}
check(
    f"Consistent feature count  ({', '.join(f'{n}={c}' for n, c in col_counts.items())})",
    len(set(col_counts.values())) == 1,
)

# 4. X_test_proc has exactly 6000 rows
check(
    f"X_test_proc row count == 6000  (actual={X_test_proc.shape[0]})",
    X_test_proc.shape[0] == 6000,
)

# 5. No zero-variance feature in X_tr_res
zero_var_cols = X_tr_res.columns[X_tr_res.var() == 0].tolist()
check(
    f"No zero-variance features in X_tr_res  ({len(zero_var_cols)} found: {zero_var_cols if zero_var_cols else 'none'})",
    len(zero_var_cols) == 0,
)

# 6. Val class ratio within 2% of train (pre-SMOTE) class ratio
train_minority_rate = y_tr.mean()
val_minority_rate   = y_val.mean()
ratio_diff = abs(val_minority_rate - train_minority_rate)
check(
    f"Val/train class ratio within 2%  (train={train_minority_rate:.4f}, val={val_minority_rate:.4f}, diff={ratio_diff:.4f})",
    ratio_diff <= 0.02,
)


# 7. Column names identical between X_tr_res and X_test_proc
col_match = list(X_tr_res.columns) == list(X_test_proc.columns)
check(
    f"Column names match between X_tr_res and X_test_proc",
    col_match
)

# ── Summary ──────────────────────────────────────────────────────────────────
n_pass = sum(p for _, p in results)
n_fail = len(results) - n_pass
print(f"{'='*60}")
print(f"  {n_pass}/{len(results)} checks passed" + ("  — pipeline ready" if n_fail == 0 else f"  — {n_fail} check(s) need attention"))
print(f"{'='*60}")





✅ PASS  |  No NaNs  (X_tr_res=0, X_val=0, X_test_proc=0)
✅ PASS  |  No infinities  (X_tr_res=0, X_val=0, X_test_proc=0)
✅ PASS  |  Consistent feature count  (X_tr_res=33, X_val=33, X_test_proc=33)
✅ PASS  |  X_test_proc row count == 6000  (actual=6000)
✅ PASS  |  No zero-variance features in X_tr_res  (0 found: none)
✅ PASS  |  Val/train class ratio within 2%  (train=0.2212, val=0.2211, diff=0.0001)
✅ PASS  |  Column names match between X_tr_res and X_test_proc
  7/7 checks passed  — pipeline ready


All 7 checks passing indicates that the pipeline produced clean and structurally consistent outputs, with no obvious data quality issues introduced during preprocessing and resampling. These automated checks were run before modelling to catch silent failures such as missing values, infinite values, feature misalignment, or unexpected shape changes.

The zero-variance check is useful after preprocessing and resampling because features with no variation would add no predictive value and may signal a problem in the transformation process. No such case was found in the resampled training set.

The column alignment check is stricter than a simple feature-count check because it confirms that features appear in the same order across splits. Since scikit-learn models rely on column position, a misordered feature matrix could produce incorrect predictions without necessarily raising an error. 

Taken together, these checks provide a useful validation step showing that the processed data is ready for modelling.

## Step 7: Save Artefacts

All processed splits and fitted pipeline objects are persisted to `data/processed/artefacts/`.
Downstream modelling notebooks load these files directly so no preprocessing is repeated.

In [15]:

artefact_dir = os.path.join(os.path.dirname(os.getcwd()), "data", "processed", "artefacts")
os.makedirs(artefact_dir, exist_ok=True)

#  CSVs 
csv_map = {
    "X_tr.csv":        X_tr,
    "y_tr.csv":        y_tr.to_frame(),
    "X_tr_res.csv":    X_tr_res,
    "X_val.csv":       X_val,
    "X_test_proc.csv": X_test_proc,
    "y_tr_res.csv":    y_tr_res.to_frame(),
    "y_val.csv":       y_val.to_frame(),
    "y_test.csv":      y_test.to_frame(),
}

for fname, df in csv_map.items():
    df.to_csv(os.path.join(artefact_dir, fname), index=False)

#  Fitted pipeline objects 
joblib.dump(ct,               os.path.join(artefact_dir, "preprocessor.joblib"))
joblib.dump(scaler_util_bill, os.path.join(artefact_dir, "scaler_util_bill.joblib"))

print(f"Artefacts saved to: {artefact_dir}")
print()

#  Final summary table 
summary_rows = [
    ("X_tr_res (SMOTE)",  X_tr_res,    y_tr_res),
    ("X_val",             X_val,        y_val),
    ("X_test_proc",       X_test_proc,  y_test),
]

header = f"{'Split':<22}  {'Rows':>7}  {'Features':>9}  {'Class 0':>8}  {'Class 1':>8}  {'Default %':>10}"
print(header)
print("-" * len(header))
for name, X, y in summary_rows:
    vc = pd.Series(y).value_counts().sort_index()
    c0 = vc.get(0, 0)
    c1 = vc.get(1, 0)
    pct = c1 / len(y) * 100
    print(f"{name:<22}  {X.shape[0]:>7,}  {X.shape[1]:>9}  {c0:>8,}  {c1:>8,}  {pct:>9.2f}%")




Artefacts saved to: c:\Users\polym\Desktop\Term_2\Predictive\Assignment\Code\PA\data\processed\artefacts

Split                      Rows   Features   Class 0   Class 1   Default %
--------------------------------------------------------------------------
X_tr_res (SMOTE)         23,830         33    15,887     7,943      33.33%
X_val                     3,600         33     2,804       796      22.11%
X_test_proc               6,000         33     4,673     1,327      22.12%


The summary table confirms three key properties of a correctly executed preprocessing pipeline. The training set is rebalanced to a 67/33 class ratio using SMOTE (15,887 class 0; 7,943 class 1).

The validation and test sets retain the natural ~22% default rate, ensuring that evaluation metrics reflect real-world performance rather than the artificially rebalanced training distribution.

All three splits share exactly 33 features, with identical column names and ordering confirmed by the validation checks in Step 6, reducing the risk of dimensionality mismatches corrupting modelling. The fitted ColumnTransformer and StandardScaler are saved as joblib objects so that the same fitted transformations can be reapplied consistently without refitting.

With these fitted objects and the preprocessing code in this notebook, the processed splits can be regenerated consistently from the raw data.

In [16]:
# Comparison table for the two training strategies

summary_rows = [
    ("X_tr_res (SMOTE)",        X_tr_res, y_tr_res),
    ("X_tr_us (Undersampled)",  X_tr_us,  y_tr_us),
]

header = f"{'Split':<24}  {'Rows':>7}  {'Features':>9}  {'Class 0':>8}  {'Class 1':>8}  {'Default %':>10}"
print(header)
print("-" * len(header))

for name, X, y in summary_rows:
    vc = pd.Series(y).value_counts().sort_index()
    c0 = vc.get(0, 0)
    c1 = vc.get(1, 0)
    pct = c1 / len(y) * 100
    print(f"{name:<24}  {X.shape[0]:>7,}  {X.shape[1]:>9}  {c0:>8,}  {c1:>8,}  {pct:>9.2f}%")

Split                        Rows   Features   Class 0   Class 1   Default %
----------------------------------------------------------------------------
X_tr_res (SMOTE)           23,830         33    15,887     7,943      33.33%
X_tr_us (Undersampled)     13,539         33     9,026     4,513      33.33%


Both resampling strategies produce the same 67/33 class ratio (minority : majority ≈ 1:2), allowing models trained on each dataset to be compared under equivalent class balance conditions.

However, the mechanisms differ. SMOTE increases the minority class by generating synthetic observations, resulting in a larger training set (23,830 rows) while preserving all original majority-class samples. In contrast, random undersampling reduces the majority class, producing a smaller dataset (13,539 rows) while retaining only a subset of the original majority observations.

This difference has modelling implications. The SMOTE dataset provides more training examples but introduces synthetic samples, while the undersampled dataset avoids synthetic data but discards many majority-class observations, potentially reducing information available for learning. Both versions are therefore retained for modelling so that performance differences between oversampling and undersampling approaches can be evaluated empirically.


## Outputs for Modelling

The following processed datasets are saved to `data/processed/artefacts/` for use in `03_modelling.ipynb`:

* **`X_tr_res.csv` / `y_tr_res.csv`** — SMOTE-resampled training set (**23,830 rows**, **33/67 class ratio**). Synthetic minority samples are generated via SMOTE while the majority class remains unchanged.

* **`X_tr_us.csv` / `y_tr_us.csv`** — Random undersampled training set (**13,539 rows**, **33/67 class ratio**). All minority-class observations are retained while the majority class is randomly reduced to twice the minority count (`sampling_strategy=0.5`).

* **`X_val.csv` / `y_val.csv`** — Validation set (**3,600 rows**, natural **~22.1% default rate**). Used for model selection and threshold optimisation.

* **`X_test_proc.csv` / `y_test.csv`** — Held-out test set (**6,000 rows**, natural **~22.1% default rate**). Used only for final unbiased evaluation.

Both `X_tr_res` and `X_tr_us` are generated from the same preprocessing pipeline and therefore share identical feature columns and ordering with the validation and test sets. Column alignment and dataset integrity are verified by the automated validation checks in **Step 6** of the preprocessing notebook.

